# PassCaptcha - Text CAPTCHA OCR (CRNN + CTC Loss)
Huấn luyện mô hình CRNN nhận diện chữ/số CAPTCHA bị méo, nhiễu và gạch ngang.
- **Backend:** PyTorch + CTC Loss
- **Dữ liệu:** Tự sinh 30,000 - 50,000 mẫu bằng thư viện `captcha`
- **Đầu ra:** File ONNX tối ưu chạy cực nhanh trên CPU máy cá nhân.

In [ ]:
!pip install captcha onnx onnxruntime --quiet

In [ ]:
import os
import random
import string
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from captcha.image import ImageCaptcha
from PIL import Image
from tqdm import tqdm

CHARACTERS = string.digits + string.ascii_uppercase
NUM_CLASSES = len(CHARACTERS) + 1  # +1 cho ký tự blank của CTC
CHAR2IDX = {c: i + 1 for i, c in enumerate(CHARACTERS)}
IDX2CHAR = {i + 1: c for i, c in enumerate(CHARACTERS)}

print(f"Characters set ({len(CHARACTERS)}): {CHARACTERS}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

### 1. Dataset Sinh Ảnh Tự Động (On-the-fly Generator)

In [ ]:
class SyntheticCaptchaDataset(Dataset):
    def __init__(self, size=30000, length=5, width=160, height=60):
        self.size = size
        self.length = length
        self.generator = ImageCaptcha(width=width, height=height)
        self.width = width
        self.height = height

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        cur_len = random.choice([4, 5, 6])
        text = ''.join(random.choices(CHARACTERS, k=cur_len))
        img = self.generator.generate_image(text).convert('L')
        img = img.resize((self.width, self.height))
        img_arr = np.array(img, dtype=np.float32) / 255.0
        img_tensor = torch.tensor(img_arr).unsqueeze(0)
        
        target = torch.tensor([CHAR2IDX[c] for c in text], dtype=torch.long)
        return img_tensor, target, text

### 2. Định Nghĩa Kiến Trúc CRNN

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(CRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2, 1)),
            nn.Conv2d(512, 512, kernel_size=2, padding=0), nn.BatchNorm2d(512), nn.ReLU()
        )
        self.rnn = nn.LSTM(512, 256, bidirectional=True, num_layers=2, batch_first=True)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        feat = self.cnn(x)
        feat = feat.squeeze(2).permute(0, 2, 1)
        out, _ = self.rnn(feat)
        logits = self.fc(out)
        return logits

### 3. Vòng Lặp Huấn Luyện (Training Loop)

In [ ]:
def collate_fn(batch):
    imgs, targets, texts = zip(*batch)
    imgs = torch.stack(imgs, 0)
    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    flat_targets = torch.cat(targets)
    return imgs, flat_targets, target_lengths, texts

train_loader = DataLoader(SyntheticCaptchaDataset(size=20000), batch_size=64, shuffle=True, collate_fn=collate_fn, num_workers=2)

model = CRNN().to(DEVICE)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=8)

print("Bắt đầu huấn luyện...")
for epoch in range(1, 9):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/8")
    for imgs, flat_targets, target_lengths, _ in pbar:
        imgs = imgs.to(DEVICE)
        flat_targets = flat_targets.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(imgs)
        log_probs = logits.log_softmax(2).permute(1, 0, 2)
        
        input_lengths = torch.full(size=(imgs.size(0),), fill_value=log_probs.size(0), dtype=torch.long, device=DEVICE)
        loss = criterion(log_probs, flat_targets, input_lengths, target_lengths)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    scheduler.step()
    print(f"Epoch {epoch} hoàn tất - Loss: {total_loss / len(train_loader):.4f}")

### 4. Xuất File ONNX Tối Ưu Cho PassCaptcha

In [ ]:
model.eval()
dummy_input = torch.randn(1, 1, 60, 160).to(DEVICE)
onnx_path = "crnn_text_captcha.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=14
)
print(f"[SUCCESS] Đã xuất file mô hình: {onnx_path}")
print(f"Kích thước file: {os.path.getsize(onnx_path) / (1024*1024):.2f} MB")